# Zero Shot Prompting

## Deepseek API call

In [ ]:
import os, re, time, glob, json
from pathlib import Path
from typing import List, Tuple
from openai import OpenAI, APIConnectionError, RateLimitError, APIStatusError

In [ ]:
import os
# --- Set your API key securely in the environment (recommended) ---
# Keys are read from the environment (see .env.example); never paste them here.
os.environ.setdefault("OpenAI_API", os.environ.get("DEEPSEEK_API_KEY", ""))  # DeepSeek key
os.environ.setdefault("Mistral_API", os.environ.get("MISTRAL_API_KEY", ""))
assert os.getenv("OpenAI_API"), "Please set DEEPSEEK_API_KEY in the environment."

# Prompt folders
ZERO_DIR = "Zero_Shot_Prompts"   # contains prompt_1.txt ... prompt_10.txt
FEW_DIR  = "Few_Shot_Prompts"    # contains prompt_1.txt ... prompt_10.txt

# Output base folder
OUT_ROOT = "Outputs/DeepSeek"

# DeepSeek API config
DEEPSEEK_MODEL = "deepseek-chat"
BASE_URL = "https://api.deepseek.com"
SYSTEM_PROMPT = "You are a cybersecurity expert assisting in interpreting ML model outputs for DDoS detection."

# Inference settings
TEMPERATURE = 0.2
TOP_P = 0.95
MAX_RETRIES = 5
PER_CALL_SLEEP = 0.25   # small pause between calls

# Optional: label a second pass for consistency (e.g., "run2"). Leave "" for the first pass.
RUN_NAME = ""           # e.g., "run2"


In [ ]:
def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_prompts(folder: str) -> List[Tuple[int, str]]:
    """
    Loads prompt files, detecting both naming patterns:
      - prompt_zero_shot_<id>.txt
      - fewshot_id_<id>.txt
    Returns list of (id, full_path)
    """
    files = glob.glob(os.path.join(folder, "*.txt"))
    items = []
    for f in files:
        stem = Path(f).stem
        # Extract ID number from filename
        num = None
        for token in stem.split("_"):
            if token.isdigit():
                num = int(token)
                break
        if num is None:
            continue
        items.append((num, f))
    return sorted(items, key=lambda x: x[0])

def backoff_sleep(attempt: int):
    t = min(5 * attempt, 20)
    print(f"Retrying after {t}s ...")
    time.sleep(t)


In [ ]:
def run_deepseek_batch(
    zero_dir: str,
    few_dir: str,
    out_root: str,
    run_name: str = "",
    model: str = "deepseek-reasoner",
    system_prompt: str = SYSTEM_PROMPT,
    temperature: float = 0.2,
    top_p: float = 0.95,
    max_retries: int = 5,
    per_call_sleep: float = 0.25,
    base_url: str = BASE_URL,
):
    key = os.getenv("OpenAI_API")
    if not key:
        raise RuntimeError("API_KEY not set. Please set it in the environment.")

    client = OpenAI(api_key=key, base_url=base_url)

    zs_sub = "zero_shot" + (f"_{run_name}" if run_name else "")
    fs_sub = "few_shot"  + (f"_{run_name}" if run_name else "")

    out_zs = os.path.join(out_root, zs_sub)
    out_fs = os.path.join(out_root, fs_sub)
    ensure_dir(out_zs); ensure_dir(out_fs)

    zero_prompts = load_prompts(zero_dir)
    few_prompts  = load_prompts(few_dir)

    zs_log_path = os.path.join(out_root, f"{zs_sub}_log.csv")
    fs_log_path = os.path.join(out_root, f"{fs_sub}_log.csv")

    def write_csv_header(path: str):
        if not os.path.exists(path):
            with open(path, "w", encoding="utf-8") as f:
                f.write("prompt_id,file_in,file_out,latency_s,model,usage_input_tokens,usage_output_tokens,status,error\n")

    write_csv_header(zs_log_path)
    write_csv_header(fs_log_path)

    def call_and_save(items, mode_name, out_dir, log_path):
        for pid, in_path in items:
            out_txt   = os.path.join(out_dir, f"prompt_{pid}.txt")
            meta_json = os.path.join(out_dir, f"prompt_{pid}.meta.json")

            # Skip already done
            if os.path.exists(out_txt):
                print(f"⏩ Skipping {in_path}, already done.")
                continue

            user_prompt = read_text(in_path)
            status = "ok"
            err_text = ""
            usage_in = usage_out = ""
            t0 = time.perf_counter()

            for attempt in range(1, max_retries + 1):
                try:
                    resp = client.chat.completions.create(
                        model=model,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user",   "content": user_prompt},
                        ],
                        temperature=temperature,
                        top_p=top_p,
                        stream=False,
                    )
                    dt = time.perf_counter() - t0
                    content = resp.choices[0].message.content

                    try:
                        usage_in  = getattr(resp.usage, "prompt_tokens", "")
                        usage_out = getattr(resp.usage, "completion_tokens", "")
                    except Exception:
                        usage_in = usage_out = ""

                    # Save outputs
                    with open(out_txt, "w", encoding="utf-8") as f:
                        f.write(content)
                    meta = {
                        "model": model,
                        "mode": mode_name,
                        "prompt_id": pid,
                        "input_file": in_path,
                        "output_file": out_txt,
                        "latency_s": dt,
                        "usage": {"prompt_tokens": usage_in, "completion_tokens": usage_out}
                    }
                    with open(meta_json, "w", encoding="utf-8") as f:
                        json.dump(meta, f, indent=2)

                    with open(log_path, "a", encoding="utf-8") as logf:
                        logf.write(f"{pid},{in_path},{out_txt},{dt:.2f},{model},{usage_in},{usage_out},ok,\n")

                    print(f"✅ Prompt {pid} done ({dt:.2f}s)")
                    time.sleep(per_call_sleep)
                    break

                except (APIConnectionError, RateLimitError, APIStatusError, Exception) as e:
                    err_text = str(e).replace("\n", " ").replace(",", " ")
                    status = "error"
                    if attempt >= max_retries:
                        dt = time.perf_counter() - t0
                        with open(log_path, "a", encoding="utf-8") as logf:
                            logf.write(f"{pid},{in_path},{out_txt},{dt:.2f},{model},{usage_in},{usage_out},{status},{err_text}\n")
                        print(f"❌ Failed on prompt {pid}: {err_text}")
                        break
                    backoff_sleep(attempt)

    if zero_prompts:
        print(f"🚀 Running Model on {len(zero_prompts)} zero-shot prompts from: {zero_dir}")
        call_and_save(zero_prompts, "zero_shot", out_zs, zs_log_path)
    else:
        print(f"[warn] No zero-shot prompts in {zero_dir}")

    if few_prompts:
        print(f"🚀 Running Modle on {len(few_prompts)} few-shot prompts from: {few_dir}")
        call_and_save(few_prompts, "few_shot", out_fs, fs_log_path)
    else:
        print(f"[warn] No few-shot prompts in {few_dir}")

    print("✅ Done.")
    print("Outputs:", out_root)
    print("Logs:", zs_log_path, "and", fs_log_path)


In [ ]:
run_deepseek_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/OpenAI",
    run_name=RUN_NAME,           # "" for first pass
    model="gpt-4-turbo", # gpt-4-turbo" # "deepseek-reasoner" # deepseek-chat 
    system_prompt=SYSTEM_PROMPT,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_retries=MAX_RETRIES,
    per_call_sleep=PER_CALL_SLEEP,
    base_url="https://api.openai.com/v1" # BASE_URL, # 
)

In [ ]:
run_deepseek_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/OpenAI",
    run_name="run_2",           # "" for first pass
    model="gpt-4-turbo", # gpt-4-turbo"
    system_prompt=SYSTEM_PROMPT,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_retries=MAX_RETRIES,
    per_call_sleep=PER_CALL_SLEEP,
    base_url="https://api.openai.com/v1" # BASE_URL, #
)

In [ ]:
import os, time, json
from openai import OpenAI

def run_openai_batch_reasoning(
    zero_dir: str,
    few_dir: str,
    out_root: str,
    run_name: str = "",
    model: str = "o4-mini",
    system_prompt: str = "You are a cybersecurity expert assisting in interpreting ML model outputs for DDoS detection.",
    temperature: float = 0.2,
    top_p: float = 0.95,
    max_retries: int = 5,
    per_call_sleep: float = 0.25,
    api_key: str = None,
):
    """
    Minimal batch runner using ONLY chat.completions.create.
    For o*-series (o1/o3/o4) models, omits temperature/top_p automatically.
    Saves one .txt and one .meta.json per prompt; logs CSV per split.
    """
    key = api_key or os.getenv("OPENAI_API_KEY")
    if not key:
        raise RuntimeError("Set OPENAI_API_KEY or pass api_key=")
    client = OpenAI(api_key=key)

    # detect o-series reasoning models
    is_reasoning = model.lower().startswith(("o1", "o3", "o4"))

    # folders
    zs_sub = "zero_shot" + (f"_{run_name}" if run_name else "")
    fs_sub = "few_shot"  + (f"_{run_name}" if run_name else "")
    out_zs = os.path.join(out_root, zs_sub)
    out_fs = os.path.join(out_root, fs_sub)
    ensure_dir(out_zs); ensure_dir(out_fs)

    zero_prompts = load_prompts(zero_dir)
    few_prompts  = load_prompts(few_dir)

    def log_path(sub): return os.path.join(out_root, f"{sub}_log.csv")
    zs_log, fs_log = log_path(zs_sub), log_path(fs_sub)

    def write_header(path):
        if not os.path.exists(path):
            with open(path, "w", encoding="utf-8") as f:
                f.write("prompt_id,file_in,file_out,latency_s,model,status,error\n")
    write_header(zs_log); write_header(fs_log)

    def call_model(messages):
        # build kwargs: omit temperature/top_p for o-series
        kwargs = {"model": model, "messages": messages}
        if not is_reasoning:
            kwargs.update({"temperature": temperature, "top_p": top_p})
        resp = client.chat.completions.create(**kwargs)
        return resp.choices[0].message.content

    def process(prompts, mode_name, out_dir, log_file):
        for pid, path_in in prompts:
            out_txt = os.path.join(out_dir, f"prompt_{pid}.txt")
            meta_js = os.path.join(out_dir, f"prompt_{pid}.meta.json")
            if os.path.exists(out_txt) and os.path.exists(meta_js):
                continue

            user_text = read_text(path_in)
            t0 = time.perf_counter()

            for attempt in range(1, max_retries + 1):
                try:
                    msgs = [
                        {"role": "system", "content": system_prompt},
                        {"role": "user",   "content": user_text},
                    ]
                    text = call_model(msgs)
                    dt = time.perf_counter() - t0

                    with open(out_txt, "w", encoding="utf-8") as f:
                        f.write(text)
                    with open(meta_js, "w", encoding="utf-8") as mf:
                        json.dump({"model": model, "mode": mode_name, "latency_s": dt}, mf, indent=2)

                    with open(log_file, "a", encoding="utf-8") as lf:
                        lf.write(f"{pid},{path_in},{out_txt},{dt:.2f},{model},ok,\n")
                    time.sleep(per_call_sleep)
                    break
                except Exception as e:
                    err = str(e).replace("\n", " ")
                    if attempt == max_retries:
                        with open(log_file, "a", encoding="utf-8") as lf:
                            lf.write(f"{pid},{path_in},{out_txt},0.0,{model},error,{err}\n")
                        print(f"❌ {mode_name} prompt {pid} failed: {err}")
                    else:
                        print(f"⚠️ Retry {attempt}/{max_retries} — {err}")
                        time.sleep(min(5 * attempt, 20))

    if zero_prompts:
        print(f"🚀 Running {model} on {len(zero_prompts)} zero-shot prompts...")
        process(zero_prompts, "zero_shot", out_zs, zs_log)
    else:
        print("[warn] No zero-shot prompts found.")

    if few_prompts:
        print(f"🚀 Running {model} on {len(few_prompts)} few-shot prompts...")
        process(few_prompts, "few_shot", out_fs, fs_log)
    else:
        print("[warn] No few-shot prompts found.")

    print("✅ All done.")


In [ ]:
run_openai_batch_reasoning(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/OpenAI_reasoning",
    run_name="",
    model="o4-mini",
    system_prompt=SYSTEM_PROMPT,
    api_key=os.getenv("OpenAI_API"),
)

In [ ]:
run_openai_batch_reasoning(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/OpenAI_reasoning",
    run_name="run2",
    model="o4-mini",
    system_prompt=SYSTEM_PROMPT,
    api_key=os.getenv("OpenAI_API"),
)

In [ ]:
import pandas as pd, os

zs_log = os.path.join(OUT_ROOT, "zero_shot_log.csv")
fs_log = os.path.join(OUT_ROOT, "few_shot_log.csv")

dfs = []
for p in [zs_log, fs_log]:
    if os.path.exists(p):
        df = pd.read_csv(p)
        df["which"] = os.path.basename(p).replace("_log.csv","")
        dfs.append(df)

if dfs:
    all_logs = pd.concat(dfs, ignore_index=True)
    display(all_logs.head())

    print("\nMean latency by mode:")
    display(all_logs.groupby("which")["latency_s"].mean())

    print("\nCompleted rows by status:")
    display(all_logs["status"].value_counts())
else:
    print("No logs found yet.")


In [ ]:
import os, time, json
from google import genai

def run_gemma_batch(
    zero_dir: str,
    few_dir: str,
    out_root: str,
    run_name: str = "",
    model: str = "gemini-2.0-flash",
    system_prompt: str = "You are a cybersecurity expert assisting in interpreting ML model outputs for DDoS detection.",
    max_retries: int = 5,
    per_call_sleep: float = 0.25,
    api_key: str = None,
):
    """
    Batch-run Google Gemma / Gemini models on prompt files.
    Each run saves .txt output + .meta.json + CSV log.
    """

    key = api_key or os.getenv("GEMMA_API_KEY")
    if not key:
        raise RuntimeError("Set GEMMA_API_KEY or pass api_key=")
    client = genai.Client(api_key=key)

    # directory setup
    zs_sub = "zero_shot" + (f"_{run_name}" if run_name else "")
    fs_sub = "few_shot"  + (f"_{run_name}" if run_name else "")
    out_zs = os.path.join(out_root, zs_sub)
    out_fs = os.path.join(out_root, fs_sub)
    os.makedirs(out_zs, exist_ok=True)
    os.makedirs(out_fs, exist_ok=True)

    zero_prompts = load_prompts(zero_dir)
    few_prompts  = load_prompts(few_dir)

    def log_path(sub): return os.path.join(out_root, f"{sub}_log.csv")
    zs_log, fs_log = log_path(zs_sub), log_path(fs_sub)

    def write_header(path):
        if not os.path.exists(path):
            with open(path, "w", encoding="utf-8") as f:
                f.write("prompt_id,file_in,file_out,latency_s,model,status,error\n")
    write_header(zs_log); write_header(fs_log)

    def call_model(prompt_text: str):
        """Send content to Gemini/Gemma and return text output."""
        full_prompt = f"{system_prompt}\n\n{prompt_text}"
        resp = client.models.generate_content(
            model=model,
            contents=[full_prompt]
        )
        try:
            return resp.candidates[0].content.parts[0].text
        except Exception:
            return getattr(resp, "text", str(resp))

    def process(prompts, mode_name, out_dir, log_file):
        for pid, path_in in prompts:
            out_txt = os.path.join(out_dir, f"prompt_{pid}.txt")
            meta_js = os.path.join(out_dir, f"prompt_{pid}.meta.json")
            if os.path.exists(out_txt) and os.path.exists(meta_js):
                continue

            user_text = read_text(path_in)
            t0 = time.perf_counter()

            for attempt in range(1, max_retries + 1):
                try:
                    text = call_model(user_text)
                    dt = time.perf_counter() - t0

                    with open(out_txt, "w", encoding="utf-8") as f:
                        f.write(text)
                    with open(meta_js, "w", encoding="utf-8") as mf:
                        json.dump({"model": model, "mode": mode_name, "latency_s": dt}, mf, indent=2)

                    with open(log_file, "a", encoding="utf-8") as lf:
                        lf.write(f"{pid},{path_in},{out_txt},{dt:.2f},{model},ok,\n")

                    time.sleep(per_call_sleep)
                    break

                except Exception as e:
                    err = str(e).replace("\n", " ")
                    if attempt == max_retries:
                        with open(log_file, "a", encoding="utf-8") as lf:
                            lf.write(f"{pid},{path_in},{out_txt},0.0,{model},error,{err}\n")
                        print(f"❌ {mode_name} prompt {pid} failed: {err}")
                    else:
                        print(f"⚠️ Retry {attempt}/{max_retries} — {err}")
                        time.sleep(min(5 * attempt, 20))

    if zero_prompts:
        print(f"🚀 Running {model} on {len(zero_prompts)} zero-shot prompts...")
        process(zero_prompts, "zero_shot", out_zs, zs_log)
    else:
        print("[warn] No zero-shot prompts found.")

    if few_prompts:
        print(f"🚀 Running {model} on {len(few_prompts)} few-shot prompts...")
        process(few_prompts, "few_shot", out_fs, fs_log)
    else:
        print("[warn] No few-shot prompts found.")

    print("✅ All done.")


In [ ]:
import os
#from my_batch_scripts import run_gemma_batch  # or paste directly above

run_gemma_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/Gemma",
    run_name="",
    model="gemini-2.0-flash",
    api_key=os.environ["GOOGLE_API_KEY"],  # or set GEMMA_API_KEY in env
)


In [ ]:
import os
#from my_batch_scripts import run_gemma_batch  # or paste directly above

run_gemma_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/Gemma",
    run_name="run2",
    model="gemini-2.0-flash",
    api_key=os.environ["GOOGLE_API_KEY"],  # or set GEMMA_API_KEY in env
)


In [ ]:
# --- Mistral batch runner (Jupyter friendly) ---

import os, glob, time, json, re
from pathlib import Path
from typing import List, Tuple
from mistralai import Mistral
from datetime import datetime

# -------- helper utils --------
def ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

_num_tail = re.compile(r"(\d+)(?!.*\d)")
def _extract_id_from_name(name: str) -> int:
    # pulls the last number in the filename for sorting (e.g., ..._42.txt -> 42)
    m = _num_tail.search(name)
    return int(m.group(1)) if m else -1

def load_prompts(folder: str, pattern: str = "*.txt") -> List[Tuple[int, str]]:
    """
    Load prompts and return a sorted list of (prompt_id, filepath).
    Works with names like:
      - prompt_zero_shot_1.txt
      - fewshot_id_1.txt
      - prompt_7.txt
    """
    paths = sorted(glob.glob(os.path.join(folder, pattern)), key=lambda p: _extract_id_from_name(os.path.basename(p)))
    out = []
    for p in paths:
        pid = _extract_id_from_name(os.path.basename(p))
        if pid >= 0:
            out.append((pid, p))
    return out

def write_log_header(path: str):
    if not os.path.exists(path):
        with open(path, "w", encoding="utf-8") as f:
            f.write("timestamp,prompt_id,file_in,file_out,latency_s,model,status,error\n")

def backoff_sleep(attempt: int, base: float = 1.5, max_sec: float = 12.0):
    time.sleep(min(base * attempt, max_sec))

# -------- main runner --------
def run_mistral_batch(
    zero_dir: str,
    few_dir: str,
    out_root: str,
    run_name: str = "",
    model: str = "mistral-large-latest",
    system_prompt: str = "You are a cybersecurity expert assisting in interpreting ML model outputs for DDoS detection.",
    temperature: float = 0.2,
    top_p: float = 0.95,
    max_tokens: int = 1500,
    max_retries: int = 5,
    per_call_sleep: float = 0.2,
    api_key: str = None,
):
    key = api_key or os.getenv("Mistral_API")
    if not key:
        raise RuntimeError("Set MISTRAL_API_KEY or pass api_key=...")

    client = Mistral(api_key=key)

    # output folders + logs
    zs_sub = "zero_shot" + (f"_{run_name}" if run_name else "")
    fs_sub = "few_shot"  + (f"_{run_name}" if run_name else "")
    out_zs = os.path.join(out_root, zs_sub)
    out_fs = os.path.join(out_root, fs_sub)
    ensure_dir(out_zs); ensure_dir(out_fs)

    zs_log = os.path.join(out_root, f"{zs_sub}_log.csv")
    fs_log = os.path.join(out_root, f"{fs_sub}_log.csv")
    write_log_header(zs_log); write_log_header(fs_log)

    zero_prompts = load_prompts(zero_dir)   # handles prompt_zero_shot_*.txt
    few_prompts  = load_prompts(few_dir)    # handles fewshot_id_*.txt

    def call_and_save(items, mode_name, out_dir, log_path):
        print(f"🚀 Running {model} on {len(items)} {mode_name} prompts...")
        for pid, in_path in items:
            out_txt   = os.path.join(out_dir, f"prompt_{pid}.txt")
            meta_json = os.path.join(out_dir, f"prompt_{pid}.meta.json")
            # idempotent: skip if already exists
            if os.path.exists(out_txt) and os.path.exists(meta_json):
                continue

            user_prompt = read_text(in_path)
            t0 = time.perf_counter()
            status, err = "ok", ""
            for attempt in range(1, max_retries + 1):
                try:
                    # Mistral chat API
                    resp = client.chat.complete(
                        model=model,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user",   "content": user_prompt},
                        ],
                        temperature=temperature,
                        top_p=top_p,
                        max_tokens=max_tokens,
                    )
                    dt = time.perf_counter() - t0

                    # Extract content
                    content = ""
                    try:
                        content = resp.choices[0].message.content
                    except Exception:
                        content = str(resp)

                    # write outputs + meta
                    with open(out_txt, "w", encoding="utf-8") as f:
                        f.write(content)

                    meta = {
                        "model": model,
                        "mode": mode_name,
                        "prompt_id": pid,
                        "input_file": in_path,
                        "output_file": out_txt,
                        "latency_s": dt,
                        "temperature": temperature,
                        "top_p": top_p,
                        "max_tokens": max_tokens,
                    }
                    with open(meta_json, "w", encoding="utf-8") as f:
                        json.dump(meta, f, indent=2)

                    # log csv
                    with open(log_path, "a", encoding="utf-8") as f:
                        f.write(f"{datetime.utcnow().isoformat()},{pid},{in_path},{out_txt},{dt:.2f},{model},ok,\n")

                    time.sleep(per_call_sleep)
                    break
                except Exception as e:
                    err = str(e).replace("\n", " ").replace(",", " ")
                    status = "error"
                    if attempt >= max_retries:
                        with open(log_path, "a", encoding="utf-8") as f:
                            f.write(f"{datetime.utcnow().isoformat()},{pid},{in_path},{out_txt},0.0,{model},{status},{err}\n")
                        print(f"❌ prompt {pid} failed after {max_retries} tries: {err}")
                    else:
                        print(f"⚠️ Retry {attempt}/{max_retries} — {err}")
                        backoff_sleep(attempt)

    if zero_prompts:
        call_and_save(zero_prompts, "zero_shot", out_zs, zs_log)
    else:
        print(f"[warn] No zero-shot prompts found in: {zero_dir}")

    if few_prompts:
        call_and_save(few_prompts, "few_shot", out_fs, fs_log)
    else:
        print(f"[warn] No few-shot prompts found in: {few_dir}")

    print("✅ Done.")
    print("Outputs:", out_root)
    print("Logs:", zs_log, "and", fs_log)


In [ ]:
run_mistral_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/Mistral",
    run_name="",
    model="mistral-large-latest",   # or another Mistral chat model you use
    temperature=0.2,
    top_p=0.95,
    max_tokens=1500,
)


In [ ]:
run_mistral_batch(
    zero_dir=ZERO_DIR,
    few_dir=FEW_DIR,
    out_root="Outputs/Mistral",
    run_name="run2",
    model="mistral-large-latest",   # or another Mistral chat model you use
    temperature=0.2,
    top_p=0.95,
    max_tokens=1500,
)